In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import json
import os
from datetime import datetime
import logging
from pathlib import Path
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

/home/ayush/coding/ml_projects/fake-news/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Memory: 3.7 GB


In [3]:
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any, Optional

# --- Configuration for Datasets ---
# Centralizing paths and column names makes the code easier to update.
DATA_PATHS = {
    "liar": Path("data/raw/liar/"),
    "fakenewsnet": Path("data/raw/fakenewsnet/"),
    "indian_context": Path("data/raw/indian_context/"),
    "enriched": Path("enriched_indian_news.csv")
}

# Column names for the LIAR dataset, based on the README.
LIAR_COLUMN_NAMES = [
    'id', 'label', 'statement', 'subject', 'speaker', 'speaker_job',
    'state_info', 'party', 'bt_counts', 'false_counts', 'ht_counts',
    'mt_counts', 'pof_counts', 'context'
]


def load_liar_dataset(path: Path) -> Optional[pd.DataFrame]:
    """Loads and processes all files from the LIAR dataset directory."""
    print("Attempting to load LIAR dataset...")
    if not path.exists():
        print(f"LIAR directory not found at: {path}")
        return None

    liar_files = list(path.glob("*.tsv")) # The README indicates TSV format.
    if not liar_files:
        print(f"No .tsv files found in {path}. Note: The original dataset uses this format.")
        return None

    dfs = []
    for file in liar_files:
        # Correctly load as a Tab-Separated file (TSV) with no header.
        df = pd.read_csv(file, sep='\t', header=None, names=LIAR_COLUMN_NAMES)
        dfs.append(df)

    if not dfs:
        return None

    combined_df = pd.concat(dfs, ignore_index=True)

    # --- Vectorized Processing ---
    # Map multi-class labels to a binary format efficiently.
    true_labels = ['true', 'mostly-true', 'half-true']
    combined_df['label'] = (~combined_df['label'].isin(true_labels)).astype(int)

    # Add metadata
    combined_df['source'] = 'liar'
    combined_df['dataset_type'] = 'established'

    # Standardize column names and select final columns
    combined_df.rename(columns={'statement': 'text'}, inplace=True)

    print(f"Loaded LIAR dataset: {len(combined_df)} samples")
    return combined_df[['text', 'label', 'source', 'dataset_type']]


def load_fakenewsnet_dataset(path: Path) -> Optional[pd.DataFrame]:
    """Loads and processes the FakeNewsNet dataset."""
    print("Attempting to load FakeNewsNet dataset...")
    if not path.exists():
        print(f"FakeNewsNet directory not found at: {path}")
        return None

    files_to_load = {
        'politifact_fake.csv': 1, 'politifact_real.csv': 0,
        'gossipcop_fake.csv': 1, 'gossipcop_real.csv': 0
    }

    all_dfs = []
    for filename, label in files_to_load.items():
        filepath = path / filename
        if filepath.exists():
            df = pd.read_csv(filepath)

            # Use 'title' column and rename it for consistency
            if 'title' in df.columns:
                df.rename(columns={'title': 'text'}, inplace=True)

                # --- Vectorized Processing ---
                df.dropna(subset=['text'], inplace=True)
                df = df[df['text'].str.len() > 20].copy() # Use .copy() to avoid SettingWithCopyWarning
                df['text'] = df['text'].str.slice(0, 2000)

                df['label'] = label
                all_dfs.append(df[['text', 'label']])

    if not all_dfs:
        print("No FakeNewsNet files were found or processed.")
        return None

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df['source'] = 'fakenewsnet'
    combined_df['dataset_type'] = 'established'

    print(f"Loaded FakeNewsNet dataset: {len(combined_df)} samples")
    return combined_df


def load_indian_context_dataset(path: Path) -> Optional[pd.DataFrame]:
    """Loads and processes custom Indian context news datasets."""
    print("Attempting to load Indian Context dataset...")
    if not path.exists():
        print(f"Indian Context directory not found at: {path}")
        return None

    datasets = {
        "alt_news_articles.csv": {'label': 1, 'source': 'alt_news'},
        "reliable_news_articles.csv": {'label': 0, 'source': 'reliable_indian'}
    }

    all_dfs = []
    for filename, info in datasets.items():
        filepath = path / filename
        if filepath.exists():
            df = pd.read_csv(filepath)
            if 'text' in df.columns:
                df.dropna(subset=['text'], inplace=True)
                df = df[df['text'].str.len() > 50].copy()
                df['text'] = df['text'].str.slice(0, 2000)
                df['label'] = info['label']
                df['source'] = info['source']
                all_dfs.append(df)

    if not all_dfs:
        print("No Indian Context files were found or processed.")
        return None

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df['dataset_type'] = 'indian_context'

    print(f"Loaded Indian Context dataset: {len(combined_df)} samples")
    return combined_df[['text', 'label', 'source', 'dataset_type']]


def load_enriched_dataset(path: Path) -> Optional[pd.DataFrame]:
    """Loads the enriched dataset."""
    print("Attempting to load Enriched dataset...")
    if not path.exists():
        print(f"Enriched dataset not found at: {path}")
        return None

    df = pd.read_csv(path)
    if 'text' not in df.columns:
        print("Enriched dataset missing 'text' column.")
        return None

    df.dropna(subset=['text'], inplace=True)
    df = df[df['text'].str.len() > 50].copy()

    # --- Vectorized Processing ---
    df['text'] = df['text'].str.slice(0, 2000)
    # Convert string labels to binary integers
    df['label'] = (df['label'].str.lower() != 'reliable').astype(int)
    df['dataset_type'] = 'enriched'
    # Fill missing source values
    df['source'].fillna('enriched', inplace=True)

    print(f"Loaded Enriched dataset: {len(df)} samples")
    return df[['text', 'label', 'source', 'dataset_type']]


def load_and_combine_datasets() -> pd.DataFrame:
    """
    Load and combine all available datasets into a unified, standardized DataFrame.
    """
    print("--- Starting Dataset Loading Process ---")

    dataset_loaders = {
        "LIAR": (load_liar_dataset, DATA_PATHS["liar"]),
        "FakeNewsNet": (load_fakenewsnet_dataset, DATA_PATHS["fakenewsnet"]),
        "Indian Context": (load_indian_context_dataset, DATA_PATHS["indian_context"]),
        "Enriched": (load_enriched_dataset, DATA_PATHS["enriched"])
    }

    all_dfs = []
    for name, (loader_func, path) in dataset_loaders.items():
        try:
            df = loader_func(path)
            if df is not None and not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"🛑 Failed to load {name} dataset: {e}")

    if not all_dfs:
        raise ValueError("No datasets could be loaded! Please check data file paths and formats.")

    # Combine all loaded dataframes into one
    df_combined = pd.concat(all_dfs, ignore_index=True)

    # --- Final Statistics ---
    print("\n--- Combined Dataset Statistics ---")
    print(f"✅ Total samples: {len(df_combined):,}")
    print(f"   - Reliable (0): {(df_combined['label'] == 0).sum():,}")
    print(f"   - Unreliable (1): {(df_combined['label'] == 1).sum():,}")

    print("\n📊 Distribution by Source:")
    print(df_combined['source'].value_counts())

    print("\n📊 Distribution by Dataset Type:")
    print(df_combined['dataset_type'].value_counts())

    return df_combined

# --- Execute Data Loading ---
# This part remains the same.
df_combined = load_and_combine_datasets()

--- Starting Dataset Loading Process ---
Attempting to load LIAR dataset...
Loaded LIAR dataset: 12791 samples
Attempting to load FakeNewsNet dataset...
Loaded FakeNewsNet dataset: 22405 samples
Attempting to load Indian Context dataset...
Loaded Indian Context dataset: 1417 samples
Attempting to load Enriched dataset...
Enriched dataset not found at: enriched_indian_news.csv

--- Combined Dataset Statistics ---
✅ Total samples: 36,613
   - Reliable (0): 25,372
   - Unreliable (1): 11,241

📊 Distribution by Source:
source
fakenewsnet        22405
liar               12791
reliable_indian     1407
alt_news              10
Name: count, dtype: int64

📊 Distribution by Dataset Type:
dataset_type
established       35196
indian_context     1417
Name: count, dtype: int64


In [4]:
df_combined.shape

(36613, 4)

In [5]:
def clean_and_preprocess_text(df):
    """Clean and preprocess the text data"""

    print("Cleaning and preprocessing text data...")

    def clean_text(text):
        """Clean individual text entries"""
        if pd.isna(text):
            return ""

        text = str(text)

        # Remove HTML tags
        import re
        text = re.sub(r'<[^>]+>', '', text)

        # Remove URLs
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)

        # Remove email addresses
        text = re.sub(r'\S+@\S+', '', text)

        # Remove extra whitespaces
        text = re.sub(r'\s+', ' ', text)

        # Remove leading/trailing whitespace
        text = text.strip()

        return text

    # Apply cleaning
    df['text_cleaned'] = df['text'].apply(clean_text)

    # Remove samples with very short text (less than 20 characters)
    initial_count = len(df)
    df = df[df['text_cleaned'].str.len() >= 20].copy()

    # Remove samples with very long text (more than 3000 characters)
    df = df[df['text_cleaned'].str.len() <= 3000].copy()

    # Remove duplicates based on cleaned text
    df = df.drop_duplicates(subset=['text_cleaned']).copy()

    final_count = len(df)
    print(f"Removed {initial_count - final_count:,} samples during cleaning")
    print(f"Final dataset size: {final_count:,} samples")

    # Display text length statistics
    text_lengths = df['text_cleaned'].str.len()
    print(f"\nText Length Statistics:")
    print(f"Mean: {text_lengths.mean():.1f} characters")
    print(f"Median: {text_lengths.median():.1f} characters")
    print(f"Min: {text_lengths.min()} characters")
    print(f"Max: {text_lengths.max()} characters")

    return df

# Clean the data
df_cleaned = clean_and_preprocess_text(df_combined)
df_cleaned.to_csv("data/processed/df_cleaned_phase_2.csv", index=False)

Cleaning and preprocessing text data...
Removed 1,306 samples during cleaning
Final dataset size: 35,307 samples

Text Length Statistics:
Mean: 158.7 characters
Median: 78.0 characters
Min: 20 characters
Max: 2935 characters


In [6]:
MODEL_CONFIG = {
    "model_name": "distilbert-base-uncased",
    "max_length": 512,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "num_epochs": 3,
    "warmup_steps": 100,
    "weight_decay": 0.01
}

class NewsDataset(Dataset):
    """Custom Dataset for news articles"""

    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        # Tokenize the text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


def create_datasets_and_dataloaders(df, config):
    """Create train/validation/test splits and dataloaders"""

    print("Creating datasets and dataloaders...")

    # Initialize tokenizer
    tokenizer = DistilBertTokenizer.from_pretrained(config["model_name"])

    # Prepare features and labels
    texts = df['text_cleaned'].tolist()
    labels = df['label'].tolist()

    # Split data: 70% train, 15% validation, 15% test
    X_temp, X_test, y_temp, y_test = train_test_split(
        texts, labels,
        test_size=0.15,
        random_state=42,
        stratify=labels
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=0.176,  # 0.15 / (1 - 0.15) ≈ 0.176 to get 15% of original
        random_state=42,
        stratify=y_temp
    )

    print(f"Train set: {len(X_train):,} samples")
    print(f"Validation set: {len(X_val):,} samples")
    print(f"Test set: {len(X_test):,} samples")

    # Create datasets
    train_dataset = NewsDataset(X_train, y_train, tokenizer, config["max_length"])
    val_dataset = NewsDataset(X_val, y_val, tokenizer, config["max_length"])
    test_dataset = NewsDataset(X_test, y_test, tokenizer, config["max_length"])

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )

    # Print label distribution for each split
    print(f"\nLabel Distribution:")
    print(f"Train - Reliable: {y_train.count(0):,}, Unreliable: {y_train.count(1):,}")
    print(f"Validation - Reliable: {y_val.count(0):,}, Unreliable: {y_val.count(1):,}")
    print(f"Test - Reliable: {y_test.count(0):,}, Unreliable: {y_test.count(1):,}")

    return {
        'train_loader': train_loader,
        'val_loader': val_loader,
        'test_loader': test_loader,
        'tokenizer': tokenizer,
        'train_size': len(X_train),
        'val_size': len(X_val),
        'test_size': len(X_test)
    }

# Create datasets and dataloaders
data_loaders = create_datasets_and_dataloaders(df_cleaned, MODEL_CONFIG)

Creating datasets and dataloaders...
Train set: 24,728 samples
Validation set: 5,282 samples
Test set: 5,297 samples

Label Distribution:
Train - Reliable: 17,150, Unreliable: 7,578
Validation - Reliable: 3,663, Unreliable: 1,619
Test - Reliable: 3,674, Unreliable: 1,623


In [7]:
def setup_model_and_optimizer(config, train_size):
    """Setup the DistilBERT model, optimizer, and scheduler"""

    print("Setting up model, optimizer, and scheduler...")

    # Load pre-trained DistilBERT model
    model = DistilBertForSequenceClassification.from_pretrained(
        config["model_name"],
        num_labels=2,  # Binary classification
        output_attentions=False,
        output_hidden_states=False
    )

    # Move model to device
    model = model.to(device)

    # Print model information
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Model loaded: {config['model_name']}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Setup optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    # Calculate total training steps
    steps_per_epoch = train_size // config["batch_size"]
    total_training_steps = steps_per_epoch * config["num_epochs"]

    # Setup learning rate scheduler
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config["warmup_steps"],
        num_training_steps=total_training_steps
    )

    print(f"Optimizer: AdamW (lr={config['learning_rate']}, weight_decay={config['weight_decay']})")
    print(f"Scheduler: Linear with warmup ({config['warmup_steps']} warmup steps)")
    print(f"Total training steps: {total_training_steps:,}")

    return model, optimizer, scheduler

# Setup model components
model, optimizer, scheduler = setup_model_and_optimizer(MODEL_CONFIG, data_loaders['train_size'])

Setting up model, optimizer, and scheduler...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: distilbert-base-uncased
Total parameters: 66,955,010
Trainable parameters: 66,955,010
Optimizer: AdamW (lr=2e-05, weight_decay=0.01)
Scheduler: Linear with warmup (100 warmup steps)
Total training steps: 4,635


In [8]:
class MetricsCalculator:
    """Class to calculate and store training metrics"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.predictions = []
        self.true_labels = []
        self.losses = []

    def update(self, preds, labels, loss=None):
        """Update metrics with batch results"""
        # Convert predictions to numpy if they're tensors
        if torch.is_tensor(preds):
            preds = preds.cpu().numpy()
        if torch.is_tensor(labels):
            labels = labels.cpu().numpy()

        self.predictions.extend(preds)
        self.true_labels.extend(labels)

        if loss is not None:
            self.losses.append(loss)

    def compute_metrics(self):
        """Calculate final metrics"""
        # Convert predictions to class labels
        pred_labels = np.array(self.predictions)
        true_labels = np.array(self.true_labels)

        # Handle both probabilities and logits
        if pred_labels.ndim > 1:
            pred_labels = np.argmax(pred_labels, axis=1)

        # Calculate metrics
        accuracy = accuracy_score(true_labels, pred_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, pred_labels, average='weighted'
        )

        # Calculate per-class metrics
        precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
            true_labels, pred_labels, average=None
        )

        avg_loss = np.mean(self.losses) if self.losses else 0.0

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'loss': avg_loss,
            'precision_per_class': precision_per_class,
            'recall_per_class': recall_per_class,
            'f1_per_class': f1_per_class
        }

def save_checkpoint(model, optimizer, scheduler, epoch, metrics, checkpoint_path):
    """Save model checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics,
        'config': MODEL_CONFIG
    }

    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None):
    """Load model checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if scheduler:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    return checkpoint['epoch'], checkpoint['metrics']

# Create directories for saving models and logs
os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("Training utilities and metrics setup completed.")

Training utilities and metrics setup completed.


In [9]:
def train_model(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device):
    """Complete training loop with validation"""

    print("Starting model training...")
    print("=" * 60)

    # Training history
    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_f1': [],
        'learning_rates': []
    }

    best_val_f1 = 0.0
    best_epoch = 0

    # Training loop
    for epoch in range(num_epochs):
        epoch_start_time = datetime.now()
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print("-" * 40)

        # Training phase
        model.train()
        train_metrics = MetricsCalculator()

        train_pbar = tqdm(train_loader, desc="Training", leave=False)
        for batch_idx, batch in enumerate(train_pbar):
            # Move batch to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Zero gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            # Backward pass
            loss.backward()

            # Clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Update weights
            optimizer.step()
            scheduler.step()

            # Calculate predictions
            predictions = torch.nn.functional.softmax(logits, dim=-1)
            predicted_classes = torch.argmax(predictions, dim=-1)

            # Update metrics
            train_metrics.update(
                predicted_classes.cpu().numpy(),
                labels.cpu().numpy(),
                loss.item()
            )

            # Update progress bar
            current_lr = scheduler.get_last_lr()[0]
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'LR': f'{current_lr:.2e}'
            })

        # Calculate training metrics
        train_results = train_metrics.compute_metrics()

        # Validation phase
        model.eval()
        val_metrics = MetricsCalculator()

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc="Validation", leave=False)
            for batch in val_pbar:
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                # Forward pass
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                logits = outputs.logits

                # Calculate predictions
                predictions = torch.nn.functional.softmax(logits, dim=-1)
                predicted_classes = torch.argmax(predictions, dim=-1)

                # Update metrics
                val_metrics.update(
                    predicted_classes.cpu().numpy(),
                    labels.cpu().numpy(),
                    loss.item()
                )

                val_pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

        # Calculate validation metrics
        val_results = val_metrics.compute_metrics()

        # Store history
        current_lr = scheduler.get_last_lr()[0]
        history['train_loss'].append(train_results['loss'])
        history['train_accuracy'].append(train_results['accuracy'])
        history['val_loss'].append(val_results['loss'])
        history['val_accuracy'].append(val_results['accuracy'])
        history['val_f1'].append(val_results['f1'])
        history['learning_rates'].append(current_lr)

        # Print epoch results
        epoch_time = datetime.now() - epoch_start_time
        print(f"Epoch {epoch + 1} Results:")
        print(f"  Train Loss: {train_results['loss']:.4f}, Train Acc: {train_results['accuracy']:.4f}")
        print(f"  Val Loss: {val_results['loss']:.4f}, Val Acc: {val_results['accuracy']:.4f}")
        print(f"  Val F1: {val_results['f1']:.4f}, Val Precision: {val_results['precision']:.4f}")
        print(f"  Val Recall: {val_results['recall']:.4f}")
        print(f"  Learning Rate: {current_lr:.2e}")
        print(f"  Epoch Time: {epoch_time}")

        # Save checkpoint if best validation F1
        if val_results['f1'] > best_val_f1:
            best_val_f1 = val_results['f1']
            best_epoch = epoch + 1

            checkpoint_path = f"models/checkpoints/best_model_epoch_{epoch+1}.pt"
            save_checkpoint(
                model, optimizer, scheduler, epoch + 1,
                val_results, checkpoint_path
            )
            print(f"  * New best model saved! (F1: {best_val_f1:.4f})")

        print()

    print("=" * 60)
    print("Training completed!")
    print(f"Best validation F1: {best_val_f1:.4f} at epoch {best_epoch}")

    return history, best_epoch

In [10]:
training_history, best_epoch = train_model(
    model=model,
    train_loader=data_loaders['train_loader'],
    val_loader=data_loaders['val_loader'],
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=MODEL_CONFIG['num_epochs'],
    device=device
)

Starting model training...
Epoch 1/3
----------------------------------------


Epoch 1 Results:
  Train Loss: 0.4913, Train Acc: 0.7479
  Val Loss: 0.4613, Val Acc: 0.7647
  Val F1: 0.7519, Val Precision: 0.7532
  Val Recall: 0.7647
  Learning Rate: 1.36e-05
  Epoch Time: 0:24:09.151993
Checkpoint saved: models/checkpoints/best_model_epoch_1.pt
  * New best model saved! (F1: 0.7519)

Epoch 2/3
----------------------------------------


Epoch 2 Results:
  Train Loss: 0.3857, Train Acc: 0.8169
  Val Loss: 0.4666, Val Acc: 0.7768
  Val F1: 0.7613, Val Precision: 0.7676
  Val Recall: 0.7768
  Learning Rate: 6.80e-06
  Epoch Time: 0:24:08.273602
Checkpoint saved: models/checkpoints/best_model_epoch_2.pt
  * New best model saved! (F1: 0.7613)

Epoch 3/3
----------------------------------------


Epoch 3 Results:
  Train Loss: 0.2964, Train Acc: 0.8670
  Val Loss: 0.5147, Val Acc: 0.7736
  Val F1: 0.7707, Val Precision: 0.7689
  Val Recall: 0.7736
  Learning Rate: 0.00e+00
  Epoch Time: 0:24:09.012810
Checkpoint saved: models/checkpoints/best_model_epoch_3.pt
  * New best model saved! (F1: 0.7707)

Training completed!
Best validation F1: 0.7707 at epoch 3


In [10]:
def evaluate_model_on_test_set(model, test_loader, device, tokenizer):
    """Comprehensive evaluation on test set"""

    print("Evaluating model on test set...")
    print("=" * 50)

    # Load best model checkpoint
    checkpoint_files = list(Path("models/checkpoints").glob("best_model_*.pt"))
    if checkpoint_files:
        latest_checkpoint = max(checkpoint_files, key=os.path.getctime)
        print(f"Loading best model: {latest_checkpoint}")

        checkpoint = torch.load(latest_checkpoint, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])

    model.eval()
    test_metrics = MetricsCalculator()
    all_predictions_prob = []

    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for batch in test_pbar:
            # Move batch to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            # Calculate predictions
            predictions_prob = torch.nn.functional.softmax(logits, dim=-1)
            predicted_classes = torch.argmax(predictions_prob, dim=-1)

            # Store predictions
            test_metrics.update(
                predicted_classes.cpu().numpy(),
                labels.cpu().numpy(),
                loss.item()
            )

            # Store probabilities for AUC calculation
            all_predictions_prob.extend(predictions_prob[:, 1].cpu().numpy())  # Probability of class 1

    # Calculate comprehensive metrics
    test_results = test_metrics.compute_metrics()

    # Calculate AUC-ROC
    try:
        auc_score = roc_auc_score(test_metrics.true_labels, all_predictions_prob)
        test_results['auc_roc'] = auc_score
    except Exception as e:
        print(f"Could not calculate AUC-ROC: {e}")
        test_results['auc_roc'] = None

    # Print detailed results
    print("Test Set Results:")
    print("-" * 30)
    print(f"Accuracy: {test_results['accuracy']:.4f}")
    print(f"Precision: {test_results['precision']:.4f}")
    print(f"Recall: {test_results['recall']:.4f}")
    print(f"F1-Score: {test_results['f1']:.4f}")
    print(f"Loss: {test_results['loss']:.4f}")
    if test_results['auc_roc']:
        print(f"AUC-ROC: {test_results['auc_roc']:.4f}")

    # Per-class metrics
    print(f"\nPer-Class Metrics:")
    print(f"Class 0 (Reliable) - Precision: {test_results['precision_per_class'][0]:.4f}, "
          f"Recall: {test_results['recall_per_class'][0]:.4f}, "
          f"F1: {test_results['f1_per_class'][0]:.4f}")
    print(f"Class 1 (Unreliable) - Precision: {test_results['precision_per_class'][1]:.4f}, "
          f"Recall: {test_results['recall_per_class'][1]:.4f}, "
          f"F1: {test_results['f1_per_class'][1]:.4f}")

    # Detailed classification report
    print(f"\nDetailed Classification Report:")
    print(classification_report(
        test_metrics.true_labels,
        test_metrics.predictions,
        target_names=['Reliable', 'Unreliable'],
        digits=4
    ))

    # Save test results
    results_path = "logs/test_results.json"
    with open(results_path, 'w') as f:
        # Convert numpy arrays to lists for JSON serialization
        test_results_json = {
            k: v.tolist() if isinstance(v, np.ndarray) else v
            for k, v in test_results.items()
        }
        json.dump(test_results_json, f, indent=2)

    print(f"\nTest results saved to: {results_path}")

    return test_results, test_metrics.true_labels, test_metrics.predictions, all_predictions_prob

# Evaluate on test set
test_results, y_true, y_pred, y_prob = evaluate_model_on_test_set(
    model, data_loaders['test_loader'], device, data_loaders['tokenizer']
)

Evaluating model on test set...
Loading best model: models/checkpoints/best_model_epoch_3.pt


Testing: 100%|██████████| 332/332 [01:32<00:00,  3.58it/s]

Test Set Results:
------------------------------
Accuracy: 0.7795
Precision: 0.7743
Recall: 0.7795
F1-Score: 0.7761
Loss: 0.5113
AUC-ROC: 0.8398

Per-Class Metrics:
Class 0 (Reliable) - Precision: 0.8268, Recall: 0.8628, F1: 0.8444
Class 1 (Unreliable) - Precision: 0.6555, Recall: 0.5909, F1: 0.6215

Detailed Classification Report:
              precision    recall  f1-score   support

    Reliable     0.8268    0.8628    0.8444      3674
  Unreliable     0.6555    0.5909    0.6215      1623

    accuracy                         0.7795      5297
   macro avg     0.7412    0.7269    0.7330      5297
weighted avg     0.7743    0.7795    0.7761      5297


Test results saved to: logs/test_results.json


In [11]:
def save_final_model(model, tokenizer, config, test_results):
    """Save the final trained model and associated files"""

    print("Saving final model and artifacts...")

    # Create final model directory
    model_save_path = Path("models/final_fake_news_detector")
    model_save_path.mkdir(parents=True, exist_ok=True)

    # Save model and tokenizer
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    print(f"Model saved to: {model_save_path}")

    # Save configuration
    config_path = model_save_path / "training_config.json"
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)

    # Save model performance summary
    performance_summary = {
        "model_name": config["model_name"],
        "training_date": datetime.now().isoformat(),
        "test_accuracy": float(test_results["accuracy"]),
        "test_f1": float(test_results["f1"]),
        "test_precision": float(test_results["precision"]),
        "test_recall": float(test_results["recall"]),
        "auc_roc": float(test_results["auc_roc"]) if test_results["auc_roc"] else None,
        "training_config": config
    }

    summary_path = model_save_path / "performance_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(performance_summary, f, indent=2)

    print(f"Performance summary saved to: {summary_path}")

    # Create a simple inference script
    inference_script = '''
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

class FakeNewsDetector:
    def __init__(self, model_path):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = DistilBertTokenizer.from_pretrained(model_path)
        self.model = DistilBertForSequenceClassification.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()

    def predict(self, text, return_probability=True):
        """Predict if news text is reliable or unreliable"""

        # Tokenize input
        inputs = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=512,
            return_tensors='pt'
        )

        # Move to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Make prediction
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

        # Get results
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_class].item()

        label = "Reliable" if predicted_class == 0 else "Unreliable"

        if return_probability:
            return {
                "label": label,
                "confidence": confidence,
                "probabilities": {
                    "reliable": predictions[0][0].item(),
                    "unreliable": predictions[0][1].item()
                }
            }
        else:
            return label

# Example usage:
# detector = FakeNewsDetector("models/final_fake_news_detector")
# result = detector.predict("Your news text here")
# print(result)
'''

    inference_path = model_save_path / "inference_example.py"
    with open(inference_path, 'w') as f:
        f.write(inference_script)

    print(f"Inference script saved to: {inference_path}")

    # Create requirements file for deployment
    requirements = '''
torch>=1.9.0
transformers>=4.21.0
numpy>=1.21.0
pandas>=1.3.0
scikit-learn>=1.0.0
'''

    requirements_path = model_save_path / "requirements.txt"
    with open(requirements_path, 'w') as f:
        f.write(requirements.strip())

    print(f"Requirements file saved to: {requirements_path}")

    print("\nModel deployment package created successfully!")
    print(f"All files saved in: {model_save_path}")

    return model_save_path

# Save final model
final_model_path = save_final_model(model, data_loaders['tokenizer'], MODEL_CONFIG, test_results)

Saving final model and artifacts...
Model saved to: models/final_fake_news_detector
Performance summary saved to: models/final_fake_news_detector/performance_summary.json
Inference script saved to: models/final_fake_news_detector/inference_example.py
Requirements file saved to: models/final_fake_news_detector/requirements.txt

Model deployment package created successfully!
All files saved in: models/final_fake_news_detector


In [12]:
def test_model_inference(model_path):
    """Test the saved model with sample predictions"""

    print("Testing model inference with sample texts...")
    print("=" * 50)

    # Load the saved model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = DistilBertTokenizer.from_pretrained(model_path)
    model = DistilBertForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()

    def predict_text(text):
        """Make prediction for a single text"""
        inputs = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=512,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
            predicted_class = torch.argmax(predictions, dim=-1).item()
            confidence = predictions[0][predicted_class].item()

        label = "Reliable" if predicted_class == 0 else "Unreliable"
        return label, confidence, predictions[0].cpu().numpy()

    # Test samples
    test_samples = [
        {
            "text": "Scientists at Harvard University have published a peer-reviewed study showing the effectiveness of a new COVID-19 treatment in reducing hospitalizations by 30%.",
            "expected": "Reliable"
        },
        {
            "text": "SHOCKING: Government hides TRUTH about aliens! Secret documents reveal extraterrestrial contact! Click here to see what THEY don't want you to know!!!",
            "expected": "Unreliable"
        },
        {
            "text": "The Reserve Bank of India announced a 0.25% increase in the repo rate following their monetary policy committee meeting, citing inflation concerns.",
            "expected": "Reliable"
        },
        {
            "text": "Miracle cure discovered! This one weird trick doctors HATE will cure all diseases instantly! Big pharma trying to suppress this amazing discovery!",
            "expected": "Unreliable"
        },
        {
            "text": "The Indian Space Research Organisation (ISRO) successfully launched its latest satellite mission to study climate change impacts on monsoon patterns.",
            "expected": "Reliable"
        }
    ]

    print("Sample Predictions:")
    print("-" * 30)

    correct_predictions = 0
    for i, sample in enumerate(test_samples, 1):
        label, confidence, probs = predict_text(sample["text"])
        is_correct = "✓" if label == sample["expected"] else "✗"
        if label == sample["expected"]:
            correct_predictions += 1

        print(f"\nSample {i}: {is_correct}")
        print(f"Text: {sample['text'][:100]}...")
        print(f"Expected: {sample['expected']}")
        print(f"Predicted: {label} (confidence: {confidence:.3f})")
        print(f"Probabilities: Reliable={probs[0]:.3f}, Unreliable={probs[1]:.3f}")

    accuracy = correct_predictions / len(test_samples)
    print(f"\nSample Accuracy: {correct_predictions}/{len(test_samples)} ({accuracy:.1%})")

    return accuracy

# Test inference
sample_accuracy = test_model_inference(final_model_path)

Testing model inference with sample texts...
Sample Predictions:
------------------------------

Sample 1: ✓
Text: Scientists at Harvard University have published a peer-reviewed study showing the effectiveness of a...
Expected: Reliable
Predicted: Reliable (confidence: 0.788)
Probabilities: Reliable=0.788, Unreliable=0.212

Sample 2: ✓
Text: SHOCKING: Government hides TRUTH about aliens! Secret documents reveal extraterrestrial contact! Cli...
Expected: Unreliable
Predicted: Unreliable (confidence: 0.984)
Probabilities: Reliable=0.016, Unreliable=0.984

Sample 3: ✗
Text: The Reserve Bank of India announced a 0.25% increase in the repo rate following their monetary polic...
Expected: Reliable
Predicted: Unreliable (confidence: 0.598)
Probabilities: Reliable=0.402, Unreliable=0.598

Sample 4: ✓
Text: Miracle cure discovered! This one weird trick doctors HATE will cure all diseases instantly! Big pha...
Expected: Unreliable
Predicted: Unreliable (confidence: 0.981)
Probabilities: Reliabl

In [ ]:
def create_performance_visualizations(history, y_true, y_pred, y_prob, test_results):
    """Create comprehensive performance visualizations"""

    print("Creating performance visualizations...")

    # Set up the plotting style
    plt.style.use('default')
    sns.set_palette("husl")

    # Create figure with subplots
    fig = plt.figure(figsize=(20, 15))

    # 1. Training History - Loss
    plt.subplot(3, 4, 1)
    plt.plot(range(1, len(history['train_loss']) + 1), history['train_loss'], 'b-', label='Training Loss')
    plt.plot(range(1, len(history['val_loss']) + 1), history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 2. Training History - Accuracy
    plt.subplot(3, 4, 2)
    plt.plot(range(1, len(history['train_accuracy']) + 1), history['train_accuracy'], 'b-', label='Training Accuracy')
    plt.plot(range(1, len(history['val_accuracy']) + 1), history['val_accuracy'], 'r-', label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 3. Learning Rate Schedule
    plt.subplot(3, 4, 3)
    plt.plot(range(1, len(history['learning_rates']) + 1), history['learning_rates'], 'g-')
    plt.title('Learning Rate Schedule')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)

    # 4. Validation F1 Score
    plt.subplot(3, 4, 4)
    plt.plot(range(1, len(history['val_f1']) + 1), history['val_f1'], 'purple', marker='o')
    plt.title('Validation F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.grid(True, alpha=0.3)

    # 5. Confusion Matrix
    plt.subplot(3, 4, 5)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Reliable', 'Unreliable'],
                yticklabels=['Reliable', 'Unreliable'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    # 6. ROC Curve
    plt.subplot(3, 4, 6)
    if test_results.get('auc_roc') is not None:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        plt.plot(fpr, tpr, 'b-', label=f'ROC Curve (AUC = {test_results["auc_roc"]:.3f})')
        plt.plot([0, 1], [0, 1], 'r--', label='Random Classifier')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.grid(True, alpha=0.3)
    else:
        plt.text(0.5, 0.5, 'ROC Curve\nNot Available', ha='center', va='center', fontsize=12)
        plt.title('ROC Curve')

    # 7. Prediction Distribution
    plt.subplot(3, 4, 7)
    reliable_probs = [y_prob[i] for i in range(len(y_true)) if y_true[i] == 0]
    unreliable_probs = [y_prob[i] for i in range(len(y_true)) if y_true[i] == 1]

    plt.hist(reliable_probs, bins=30, alpha=0.7, label='Reliable News', color='blue')
    plt.hist(unreliable_probs, bins=30, alpha=0.7, label='Unreliable News', color='red')
    plt.xlabel('Predicted Probability (Unreliable)')
    plt.ylabel('Frequency')
    plt.title('Prediction Probability Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 8. Class-wise Performance Metrics
    plt.subplot(3, 4, 8)
    metrics = ['Precision', 'Recall', 'F1-Score']
    reliable_scores = [test_results['precision_per_class'][0],
                      test_results['recall_per_class'][0],
                      test_results['f1_per_class'][0]]
    unreliable_scores = [test_results['precision_per_class'][1],
                        test_results['recall_per_class'][1],
                        test_results['f1_per_class'][1]]

    x = np.arange(len(metrics))
    width = 0.35

    plt.bar(x - width/2, reliable_scores, width, label='Reliable', color='blue', alpha=0.7)
    plt.bar(x + width/2, unreliable_scores, width, label='Unreliable', color='red', alpha=0.7)

    plt.xlabel('Metrics')
    plt.ylabel('Score')
    plt.title('Class-wise Performance')
    plt.xticks(x, metrics)
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 9. Model Architecture Summary (Text)
    plt.subplot(3, 4, 9)
    plt.axis('off')
    architecture_text = f'''Model Architecture Summary:

• Base Model: {MODEL_CONFIG["model_name"]}
• Max Sequence Length: {MODEL_CONFIG["max_length"]}
• Number of Labels: 2 (Binary)
• Total Parameters: ~67M
• Trainable Parameters: ~67M

Training Configuration:
• Batch Size: {MODEL_CONFIG["batch_size"]}
• Learning Rate: {MODEL_CONFIG["learning_rate"]}
• Epochs: {MODEL_CONFIG["num_epochs"]}
• Optimizer: AdamW
• Scheduler: Linear with Warmup'''

    plt.text(0.05, 0.95, architecture_text, transform=plt.gca().transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace')
    plt.title('Model Configuration')

    # 10. Performance Summary (Text)
    plt.subplot(3, 4, 10)
    plt.axis('off')
    performance_text = f'''Test Set Performance:

• Accuracy: {test_results["accuracy"]:.4f}
• Precision: {test_results["precision"]:.4f}
• Recall: {test_results["recall"]:.4f}
• F1-Score: {test_results["f1"]:.4f}
• AUC-ROC: {test_results.get("auc_roc", "N/A"):.4f if test_results.get("auc_roc") else "N/A"}

Class Performance:
• Reliable: P={test_results["precision_per_class"][0]:.3f}, R={test_results["recall_per_class"][0]:.3f}
• Unreliable: P={test_results["precision_per_class"][1]:.3f}, R={test_results["recall_per_class"][1]:.3f}

Dataset Information:
• Total Samples: {len(y_true):,}
• Class Distribution: {(np.array(y_true) == 0).sum()}/{(np.array(y_true) == 1).sum()}'''

    plt.text(0.05, 0.95, performance_text, transform=plt.gca().transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace')
    plt.title('Performance Summary')

    # 11. Training Progress Over Time
    plt.subplot(3, 4, 11)
    epochs = range(1, len(history['train_loss']) + 1)
    plt.plot(epochs, history['train_loss'], 'b-', alpha=0.7, label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'r-', alpha=0.7, label='Val Loss')

    # Add secondary y-axis for accuracy
    ax2 = plt.gca().twinx()
    ax2.plot(epochs, history['train_accuracy'], 'b--', alpha=0.5, label='Train Acc')
    ax2.plot(epochs, history['val_accuracy'], 'r--', alpha=0.5, label='Val Acc')

    plt.xlabel('Epoch')
    plt.ylabel('Loss', color='black')
    ax2.set_ylabel('Accuracy', color='gray')
    plt.title('Training Progress')
    plt.grid(True, alpha=0.3)

    # 12. Error Analysis
    plt.subplot(3, 4, 12)
    # Calculate classification errors
    errors = np.array(y_true) != np.array(y_pred)
    error_rate_by_confidence = []
    confidence_bins = np.linspace(0.5, 1.0, 10)

    for i in range(len(confidence_bins) - 1):
        low, high = confidence_bins[i], confidence_bins[i + 1]
        mask = (np.maximum(np.array(y_prob), 1 - np.array(y_prob)) >= low) & \
               (np.maximum(np.array(y_prob), 1 - np.array(y_prob)) < high)
        if mask.sum() > 0:
            error_rate = errors[mask].mean()
        else:
            error_rate = 0
        error_rate_by_confidence.append(error_rate)

    bin_centers = (confidence_bins[:-1] + confidence_bins[1:]) / 2
    plt.bar(bin_centers, error_rate_by_confidence, width=0.04, alpha=0.7, color='orange')
    plt.xlabel('Prediction Confidence')
    plt.ylabel('Error Rate')
    plt.title('Error Rate vs Confidence')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()

    # Save the comprehensive plot
    plt.savefig('logs/comprehensive_performance_analysis.png', dpi=300, bbox_inches='tight')
    print("Comprehensive performance plot saved to: logs/comprehensive_performance_analysis.png")

    plt.show()

    # Save training history
    history_path = 'logs/training_history.json'
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Training history saved to: {history_path}")

# Create visualizations
create_performance_visualizations(training_history, y_true, y_pred, y_prob, test_results)